# 04 — Platform Concentration & Final Results

This notebook computes the **platform-level HHI** on cloud spending and compares it against the contractor-level baseline to quantify the **concentration gap** — the central finding of the analysis.

**Key question:** Does viewing federal IT spending through the lens of *which cloud platform* is used reveal hidden concentration that the contractor-level view obscures?

---

In [ ]:
import pandas as pd
import numpy as np
import os, importlib.util

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

def _import_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

attr_mod = _import_module('platform_attribution',
    os.path.join(PROJECT_ROOT, 'notebooks', '03_multi-stage_attribution_pipeline', 'platform_attribution.py'))
baseline_mod = _import_module('baseline_merged_hhi',
    os.path.join(PROJECT_ROOT, 'notebooks', '02_base_analysis', 'baseline_merged_hhi.py'))

## 1. Load Attributed Dataset

In [ ]:
ATTR_PATH = os.path.join(PROJECT_ROOT, 'data', '02_processed', '03_classified', 'attributed_dataset.csv')
df = pd.read_csv(ATTR_PATH)
print(f'Loaded {len(df):,} records, ${df["dollars"].sum()/1e9:.1f}B')
print(f'Cloud records: {df["is_cloud"].sum():,}')

## 2. Platform-Level HHI

In [ ]:
platform_result = attr_mod.calculate_platform_hhi(df, platform_column='final_platform')

## 3. Baseline HHI (for comparison)

In [ ]:
baseline_result = baseline_mod.calculate_contractor_hhi(df)

## 4. The Concentration Gap

The central finding: the same federal IT market looks **unconcentrated** when viewed at the contractor level but **highly concentrated** when viewed through cloud platform dependence.

In [ ]:
baseline_hhi = baseline_result['hhi']
platform_hhi = platform_result['hhi']
gap = platform_hhi / baseline_hhi if baseline_hhi > 0 else 0

def classify_hhi(hhi):
    if hhi < 1500: return 'Unconcentrated'
    elif hhi < 2500: return 'Moderately Concentrated'
    else: return 'Highly Concentrated'

print('=' * 70)
print('CONCENTRATION GAP ANALYSIS')
print('=' * 70)
print(f'''
  {"Metric":<30s} {"Baseline":>18s} {"Platform":>18s}
  {"":30s} {"(Contractor-level)":>18s} {"(Cloud-level)":>18s}
  {"-" * 66}
  {"HHI":<30s} {baseline_hhi:>18,.1f} {platform_hhi:>18,.1f}
  {"Classification":<30s} {classify_hhi(baseline_hhi):>18s} {classify_hhi(platform_hhi):>18s}
  {"C4 (%)":<30s} {baseline_result["c4"]:>18.2f} {platform_result["c4"]:>18.2f}
  {"Equivalent firms":<30s} {baseline_result["n_equivalent_firms"]:>18.1f} {platform_result["n_equivalent_firms"]:>18.1f}
  {"Entities":<30s} {baseline_result["n_contractors"]:>18,} {len(platform_result["platform_shares"]):>18,}

  CONCENTRATION GAP: {gap:.1f}x
  Platform HHI is {gap:.1f} times the contractor baseline HHI.
''')

## 5. Platform Market Shares

In [ ]:
ps = platform_result['platform_shares']
pd_dollars = platform_result['platform_spending']

print(f'Cloud spending: ${platform_result["total_cloud_dollars"]/1e9:.2f}B')
print(f'({platform_result["total_cloud_dollars"]/df["dollars"].sum()*100:.1f}% of total federal IT spending)\n')

for platform in ps.index:
    share = ps[platform]
    dollars = pd_dollars[platform]
    bar = '#' * int(share / 1.5)
    print(f'  {platform:25s} {share:6.2f}%  ${dollars/1e6:>9.1f}M  {bar}')

## 6. Temporal Trends

How has cloud spending and platform concentration changed over the study period?

In [ ]:
cloud_df = df[df['is_cloud'] == True].copy()

# Annual cloud spending by platform
annual = cloud_df.groupby(['fiscal_year', 'final_platform'])['dollars'].sum().reset_index()
annual_pivot = annual.pivot_table(index='fiscal_year', columns='final_platform',
                                   values='dollars', aggfunc='sum', fill_value=0)
annual_pivot = annual_pivot / 1e6  # Convert to millions

# Filter to main years
annual_pivot = annual_pivot.loc[annual_pivot.index.isin(range(2017, 2025))]

print('Cloud spending by platform and fiscal year ($M):')
# Show top platforms
top_platforms = platform_result['platform_shares'].head(6).index.tolist()
display_cols = [c for c in top_platforms if c in annual_pivot.columns]
annual_pivot['Total'] = annual_pivot.sum(axis=1)
print(annual_pivot[display_cols + ['Total']].round(1).to_string())

In [ ]:
# Annual HHI trend
print('\nAnnual platform HHI:')
for fy in sorted(cloud_df['fiscal_year'].dropna().unique()):
    if fy < 2017 or fy > 2024:
        continue
    fy_cloud = cloud_df[cloud_df['fiscal_year'] == fy]
    if len(fy_cloud) < 5:
        continue
    fy_spending = fy_cloud.groupby('final_platform')['dollars'].sum()
    fy_total = fy_spending.sum()
    fy_shares = fy_spending / fy_total * 100
    fy_hhi = (fy_shares ** 2).sum()
    fy_top = fy_shares.idxmax()
    bar = '#' * int(fy_hhi / 200)
    print(f'  FY{int(fy)}:  HHI {fy_hhi:>7,.0f}  ({classify_hhi(fy_hhi):25s})  '
          f'Top: {fy_top:15s} ({fy_shares.max():.1f}%)  '
          f'${fy_total/1e6:.0f}M  {bar}')

## 7. Subcontract Flows to Platforms

One of the key findings from the merged approach: subcontracts flowing directly to platform vendors.

In [ ]:
# Subcontract records that are cloud and attributed to a platform
sub_cloud = cloud_df[cloud_df['record_type'] == 'subcontract'].copy()
print(f'Cloud subcontract records: {len(sub_cloud):,}')
print(f'Cloud subcontract spending: ${sub_cloud["dollars"].sum()/1e6:.1f}M')

known_platforms = {'AWS', 'Azure', 'Google Cloud', 'Salesforce', 'Oracle Cloud',
                   'IBM Cloud', 'Multi-cloud'}

if len(sub_cloud) > 0:
    print(f'\nSubcontract cloud spending by platform:')
    sub_platforms = sub_cloud.groupby('final_platform')['dollars'].sum().sort_values(ascending=False)
    for platform, dollars in sub_platforms.items():
        n = len(sub_cloud[sub_cloud['final_platform'] == platform])
        print(f'  {platform:25s} ${dollars/1e6:>8.1f}M  ({n:,} records)')

    # Who are the prime contractors funnelling money to platforms?
    print(f'\nTop prime contractors with cloud subcontracts to known platforms:')
    sub_attributed = sub_cloud[sub_cloud['final_platform'].isin(known_platforms)]
    if len(sub_attributed) > 0:
        prime_funnel = sub_attributed.groupby('prime_contractor').agg(
            sub_dollars=('dollars', 'sum'),
            n_subs=('dollars', 'count')
        ).sort_values('sub_dollars', ascending=False).head(15)
        for prime, row in prime_funnel.iterrows():
            if pd.notna(prime):
                print(f'  {str(prime)[:45]:45s} ${row["sub_dollars"]/1e6:>8.1f}M  ({row["n_subs"]:,} subs)')

## 7b. Sensitivity Analysis: Unattributed Cloud Records

Some cloud records couldn't be attributed to a specific platform — their `final_platform` falls back to the contractor name. How sensitive are the HHI results to how these records are handled?

We test three scenarios:
1. **As-is**: Unattributed cloud records use contractor name (current approach — disperses concentration)
2. **Known platforms only**: HHI calculated only on records attributed to known platforms
3. **Proportional allocation**: Distribute unattributed dollars across known platforms in proportion to their current shares

In [ ]:
# Sensitivity analysis: how does HHI change under different allocation assumptions?

known_platforms = {'AWS', 'Azure', 'Google Cloud', 'Salesforce', 'Oracle Cloud',
                   'IBM Cloud', 'Multi-cloud'}

# Identify unattributed cloud records
unattributed_mask = (cloud_df['attribution_method'] == 'cloud_unattributed_contractor')
unattributed_dollars = cloud_df.loc[unattributed_mask, 'dollars'].sum()
total_cloud_dollars = cloud_df['dollars'].sum()

# Known platform spending
known_cloud = cloud_df[cloud_df['final_platform'].isin(known_platforms)]
known_spending = known_cloud.groupby('final_platform')['dollars'].sum().sort_values(ascending=False)
known_total = known_spending.sum()

# Scenario 1: As-is (current — unattributed dispersed across contractor names)
hhi_asis = platform_hhi

# Scenario 2: Known platforms only (exclude unattributed)
if known_total > 0:
    known_shares = known_spending / known_total * 100
    hhi_known_only = (known_shares ** 2).sum()
else:
    hhi_known_only = 0

# Scenario 3: Proportional allocation (distribute unattributed to known platforms)
if known_total > 0:
    proportions = known_spending / known_total
    allocated_spending = known_spending + proportions * unattributed_dollars
    allocated_total = allocated_spending.sum()
    allocated_shares = allocated_spending / allocated_total * 100
    hhi_proportional = (allocated_shares ** 2).sum()
else:
    hhi_proportional = 0

print('SENSITIVITY ANALYSIS: PLATFORM HHI UNDER DIFFERENT ALLOCATION SCENARIOS')
print('=' * 70)
print(f'\n  Unattributed cloud spending: ${unattributed_dollars/1e6:.1f}M '
      f'({unattributed_dollars/total_cloud_dollars*100:.1f}% of cloud)')
print(f'  Unattributed cloud records: {unattributed_mask.sum():,}')
print(f'\n  {"Scenario":<45s} {"HHI":>8s}  {"Classification":>25s}  {"Gap":>6s}')
print(f'  {"-"*87}')

for label, hhi_val in [
    ('1. As-is (contractor name fallback)', hhi_asis),
    ('2. Known platforms only', hhi_known_only),
    ('3. Proportional allocation to platforms', hhi_proportional),
]:
    gap_val = hhi_val / baseline_hhi if baseline_hhi > 0 else 0
    print(f'  {label:<45s} {hhi_val:>8,.1f}  {classify_hhi(hhi_val):>25s}  {gap_val:>5.1f}x')

# Show what proportional allocation looks like
if known_total > 0:
    print(f'\n  Known platform-only HHI ({hhi_known_only:,.1f}) represents the "floor" of')
    print(f'  concentration — what we can measure with certainty.')
    print(f'\n  Proportional allocation would distribute ${unattributed_dollars/1e6:.1f}M as:')
    for platform in proportions.sort_values(ascending=False).index:
        alloc = proportions[platform] * unattributed_dollars
        print(f'    {platform:25s} +${alloc/1e6:>7.1f}M  '
              f'(share: {allocated_shares[platform]:.1f}%)')

## 8. Final Summary

In [ ]:
total_spending = df['dollars'].sum()
cloud_spending = cloud_df['dollars'].sum()

known_platforms = {'AWS', 'Azure', 'Google Cloud', 'Salesforce', 'Oracle Cloud',
                   'IBM Cloud', 'Multi-cloud'}
platform_specified = cloud_df[cloud_df['final_platform'].isin(known_platforms)]

# Classification method
has_llm = ('classification_source' in df.columns and
           df['classification_source'].str.contains('llm', na=False).any())
cls_method = 'RegEx + LLM' if has_llm else 'RegEx-only'

print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)
print(f'''
  DATA
  Total merged records:          {len(df):>10,}
  Total spending:                ${total_spending/1e9:>9.1f}B
  Cloud records:                 {len(cloud_df):>10,}  ({len(cloud_df)/len(df)*100:.1f}%)
  Cloud spending:                ${cloud_spending/1e6:>9.1f}M  ({cloud_spending/total_spending*100:.1f}%)
  Platform specified:            {len(platform_specified):>10,}  ({len(platform_specified)/len(cloud_df)*100:.1f}% of cloud)
  Classification method:         {cls_method:>10s}

  CONCENTRATION
  Baseline HHI (contractor):     {baseline_hhi:>10,.1f}  {classify_hhi(baseline_hhi)}
  Platform HHI (cloud):          {platform_hhi:>10,.1f}  {classify_hhi(platform_hhi)}
  Concentration gap:             {gap:>10.1f}x

  INTERPRETATION
  The federal IT contractor market appears competitive (HHI {baseline_hhi:.0f}).
  However, cloud spending is highly concentrated among a few platforms
  (HHI {platform_hhi:.0f}), with the top 4 controlling {platform_result["c4"]:.1f}% of cloud dollars.
  This {gap:.0f}x concentration gap reveals hidden platform dependence
  that is invisible at the contractor level.
''')

---

**Pipeline complete.** Output files:
- `data/02_processed/01_filtered/prime_services_filtered.csv`
- `data/02_processed/02_merged/merged_dataset.csv`
- `data/02_processed/03_classified/classified_dataset.csv` (Stage 1)
- `data/02_processed/03_classified/attributed_dataset.csv` (Stage 2)